# Diagnose a vLLM setup on a free GPU

An agent sends the same instructions and tool definitions with every request. Its vLLM setup here has two deliberate problems. Tensward measures it, names what holds it back and suggests a change. You apply the first suggestion, measure again, and repeat once.

What happens in this notebook:

- The agent's setup is measured and Tensward names the bottleneck.
- The first suggested change is measured, and Tensward shows what changed.
- The next suggestion is measured the same way, and Tensward shows what that change did on its own.

The notebook downloads its inputs and two small marker files from the Tensward release, whose public download counts are how we see how many people start and finish it; nothing else is sent.

**Before you start:** Runtime → Change runtime type → T4 GPU. Then Runtime → Run all. It takes about 20 minutes.

In [ ]:
import shutil, subprocess, urllib.request
TAG = "v0.3.3"
RELEASE = f"https://github.com/Tensward/tensward/releases/download/{TAG}"
try:
    urllib.request.urlopen(f"{RELEASE}/notebook-start.txt", timeout=10).read()
except Exception:
    pass
class StopNotebook(Exception):
    """Shown as a single line; stops Run all."""
    def _render_traceback_(self):
        return [str(self)]
if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise StopNotebook("No GPU in this runtime. Choose Runtime → Change runtime type → T4 GPU, then run all again.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

## Choose a model

The default fits a T4 (16 GB) and needs no account. The times in this notebook are for the default model; larger ones take longer. To try another model, change the three settings below; Tensward then measures that model, so its findings may differ.

In [ ]:
# Validated on a T4 with this notebook:
#   Qwen/Qwen2.5-1.5B-Instruct (default): PRECISION "bf16", TOOL_PARSER "hermes"
#   Qwen/Qwen2.5-7B-Instruct-AWQ: PRECISION "int4", TOOL_PARSER "hermes"
# Other models work too when TOOL_PARSER matches the model (see vLLM's tool-calling docs)
# and the model fits the T4's 16 GB.
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PRECISION = "bf16"
TOOL_PARSER = "hermes"

## Install

Installing vLLM replaces Colab's PyTorch and takes a few minutes. pip may print dependency-conflict warnings about other preinstalled Colab packages; they do not affect this notebook. Colab's `torchaudio` is built for the old PyTorch, so the cell removes it (vLLM does not need it here). `tensward env` will say Docker is not available: that is expected on Colab, where Tensward runs vLLM directly (`--runtime local`).

The `run` helper runs a shell command, shows its output as it goes, and stops the notebook if the command fails.

In [ ]:
import pathlib, re, subprocess
def run(command: str) -> None:
    """Run a shell command, showing its output as it goes; stop the notebook if it fails."""
    print("$", command, flush=True)
    process = subprocess.Popen(
        command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise StopNotebook(f"That command failed (exit {code}); its error is above.")
run('pip -q install --progress-bar off "tensward>=0.3.3" "vllm==0.30.0"')
run("pip -q uninstall -y torchaudio")
run("tensward env")

## Download a model

The chosen model is downloaded to `/content/model`.

In [ ]:
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars, logging

disable_progress_bars()
logging.set_verbosity_error()
print(f"Downloading {MODEL}...", flush=True)
snapshot_download(MODEL, local_dir="/content/model")
print("Model downloaded to /content/model")

## Workload

A workload is a serving configuration plus a file of prompts. This one sends 128 requests with 32 clients at a time. Every prompt starts with the same long instructions and tool definitions, followed by a different employee question. The configuration is adapted to this model and to the setup below.

In [ ]:
import json, pathlib
work = pathlib.Path("/content/tensward"); work.mkdir(exist_ok=True)
urllib.request.urlretrieve(f"{RELEASE}/prompts-agent.jsonl", work / "prompts-agent.jsonl")
config = json.loads(urllib.request.urlopen(f"{RELEASE}/config.json").read())
config["case"].update({"weight_precision": PRECISION,
                       "activation_dtype": "bfloat16" if PRECISION == "bf16" else "float16",
                       "max_model_len": 4096, "max_num_seqs": 2, "gpu_memory_utilization": 0.85})
config["workload"].update({"request_count": 128, "output_tokens": 128,
                           "arrival": {"kind": "closed_loop", "concurrency": 32}})
(work / "config.json").write_text(json.dumps(config, indent=1))

## Register your current setup

The two flags stand in for your own setup's choices: `--max-num-seqs 2` lets only 2 requests run at once, and `--no-enable-prefix-caching` switches off vLLM's prefix cache. The T4 has no bf16 support, so the command serves in float16 with `--dtype float16`.

In [ ]:
import time
project = f"/content/tensward/project-{int(time.time())}"
current = (f"vllm serve /content/model --max-num-seqs 2 --no-enable-prefix-caching --dtype float16 "
           f"--max-model-len 4096 --enable-auto-tool-choice --tool-call-parser {TOOL_PARSER}")
run(f"tensward init --project {project} --model /content/model --config /content/tensward/config.json "
    f"--prompts /content/tensward/prompts-agent.jsonl --current '{current}' > /content/tensward/init.json")
print("Registered. The full record is in /content/tensward/init.json.")

## Analyse

Tensward starts your current setup, sends the workload, and reads the engine's metrics.

In [ ]:
run(f"tensward analyse --project {project} --runtime local")
def run_ids(project):
    return sorted(p.name for p in pathlib.Path(project, "runs").iterdir())
def newest_report(project):
    return pathlib.Path(project, "runs", run_ids(project)[-1], "report.md").read_text()
def section(report, *titles):
    parts = re.split(r"(?m)^## ", report)
    return "\n".join("## " + p for p in parts if p.split("\n", 1)[0].startswith(titles))
print(section(newest_report(project), "Diagnosis"))

## Try the first suggestion

The next cell takes the first `Try:` command from the report, shows the suggestion and its reason, runs it, and prints what changed.

In [ ]:
def follow():
    suggestions = section(newest_report(project), "What to try next").split("\n")
    tries = [i for i, line in enumerate(suggestions) if re.match(r"\s*Try: `tensward analyse ", line)]
    if not tries:
        raise StopNotebook("Tensward suggests nothing more for this setup.")
    first = tries[0]
    start = max(i for i in range(first) if suggestions[i].startswith("- "))
    print("\n".join(suggestions[start:first]), flush=True)
    command = re.search(r"Try: `(tensward analyse [^`]+)`", suggestions[first]).group(1)
    run(command + " --runtime local")
    report = newest_report(project)
    print(section(report, "What changed"))
    print(section(report, "Diagnosis"))
follow()

## Try the next suggestion

Fixing one bottleneck often exposes the next. The same cell follows the new report's first suggestion.

In [ ]:
follow()

## What the second change did

The report above compares with your registered setup, so it shows both changes together. This compares the second run with the first, so it shows the second change on its own.

In [ ]:
before, after = run_ids(project)[-2:]
result = subprocess.run(["tensward", "compare", "--project", project, before, after],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise StopNotebook(f"tensward compare failed (exit {result.returncode}): {result.stderr.strip()}")
print("Second run against the first:\n")
print(result.stdout.split("### Prompt")[0].strip())
for line in result.stdout.splitlines():
    if line.startswith("answers side by side"):
        print(line)

A few answers may differ in wording between setups, because float16 arithmetic depends on how requests are batched and served; compare the side-by-side file.

## Next steps

Register your own model and setup: pass `--current` with the exact command you serve with. See the README sections on [installing](https://github.com/Tensward/tensward#install), [`init`](https://github.com/Tensward/tensward#2-register-the-project) and [`analyse`](https://github.com/Tensward/tensward#3-analyse).

Found something, right or wrong? Open an issue: https://github.com/Tensward/tensward/issues

In [ ]:
try:
    urllib.request.urlopen(f"{RELEASE}/notebook-finish.txt", timeout=10).read()
except Exception:
    pass
print("Done.")